# Video estimate for Fortune 500 CEOs - USA

## Set up CEOs list

In [3]:
import json
import csv

with open("data/ceos_usa_2023.csv") as f:
    companies = list(csv.DictReader(f))

ceos_usa = [
    {
        "CEO": c["CEO"],
        "Company": c["Company"],
        "year": 2023,
        "general_videos_collected": False,
        "dei_videos_collected": False
    }
    for c in companies
]

with open("data/ceos_usa_2023.json", "w") as f:
    json.dump(ceos_usa, f, indent=2)

print(f"Wrote {len(ceos_usa)} companies")

Wrote 500 companies


In [5]:
import json
import re

with open("data/ceos_usa_2023.json") as f:
    ceos_usa = json.load(f)

def strip_middle_initial(name):
    alt = re.sub(r"\s+[A-Z]\.?\s+", " ", name)
    return alt.strip()

for c in ceos_usa:
    alt = strip_middle_initial(c["CEO"])
    c["CEO_alt"] = alt if alt != c["CEO"] else None

with open("data/ceos_usa_2023.json", "w") as f:
    json.dump(ceos_usa, f, indent=2)

In [6]:
import json

with open("data/ceos_usa_2023.json") as f:
    ceos_usa = json.load(f)

ceos_usa = [c for c in ceos_usa if c.get("CEO")]

with open("data/ceos_usa_2023.json", "w") as f:
    json.dump(ceos_usa, f, indent=2)

print(f"Remaining: {len(ceos_usa)}")

Remaining: 476


## Get videos per CEO

In [ ]:
import os
import json
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()

youtube = build(
    "youtube",
    "v3",
    developerKey=os.getenv("YOUTUBE_API_KEY_4")
)

CEOS_PATH = "data/ceos_usa_2023.json"
OUTPUT_DIR = "output/estimation/2023_USA/general"

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(CEOS_PATH) as f:
    ceo_list = json.load(f)

eligible_ceos = [c["CEO"] for c in ceo_list if not c.get("general_videos_collected")]

if len(eligible_ceos) == 0:
    print("No new CEOs left to collect.")
else:
    for ceo in eligible_ceos:
        search_response = youtube.search().list(
            part="snippet",
            q=f"{ceo} interview",
            type="video",
            publishedAfter="2023-01-01T00:00:00Z",
            publishedBefore="2024-01-01T00:00:00Z",
            maxResults=50
        ).execute()

        video_ids = [
            item["id"]["videoId"]
            for item in search_response["items"]
        ]

        details_response = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(video_ids)
        ).execute() if video_ids else {"items": []}

        output = {
            "ceo": ceo,
            "search_results": search_response,
            "video_details": details_response
        }

        with open(f"{OUTPUT_DIR}/{ceo}.json", "w") as f:
            json.dump(output, f, indent=2)

        # Update ceos_usa_2023.json immediately after success
        with open(CEOS_PATH) as f:
            ceo_list = json.load(f)

        for c in ceo_list:
            if c["CEO"] == ceo:
                c["general_videos_collected"] = True
                break

        with open(CEOS_PATH, "w") as f:
            json.dump(ceo_list, f, indent=2)

        print(f"Completed: {ceo}")

In [ ]:
import os
import json

CEOS_PATH = "data/ceos_usa_2023.json"
GENERAL_DIR = "output/estimation/2023_USA/general"
POSTPROCESSED_DIR = f"{GENERAL_DIR}/preprocessed"

os.makedirs(POSTPROCESSED_DIR, exist_ok=True)

with open(CEOS_PATH) as f:
    ceo_list = json.load(f)

alt_lookup = {c["CEO"]: c.get("CEO_alt") for c in ceo_list}

def name_in_title(name, title):
    if not name or not title:
        return False
    return name.lower() in title.lower()

for file in os.listdir(GENERAL_DIR):
    if not file.endswith(".json"):
        continue

    filepath = os.path.join(GENERAL_DIR, file)
    if not os.path.isfile(filepath):
        continue

    with open(filepath) as f:
        data = json.load(f)

    ceo = data["ceo"]
    ceo_alt = alt_lookup.get(ceo)

    original_items = data.get("video_details", {}).get("items", [])
    filtered_items = [
        video for video in original_items
        if name_in_title(ceo, video.get("snippet", {}).get("title", ""))
        or name_in_title(ceo_alt, video.get("snippet", {}).get("title", ""))
    ]

    kept_video_ids = {video["id"] for video in filtered_items}

    original_search_items = data.get("search_results", {}).get("items", [])
    filtered_search_items = [
        item for item in original_search_items
        if item.get("id", {}).get("videoId") in kept_video_ids
    ]

    new_data = dict(data)
    new_data["search_results"] = dict(data.get("search_results", {}))
    new_data["search_results"]["items"] = filtered_search_items
    new_data["video_details"] = dict(data.get("video_details", {}))
    new_data["video_details"]["items"] = filtered_items

    with open(os.path.join(POSTPROCESSED_DIR, file), "w") as f:
        json.dump(new_data, f, indent=2)

    print(f"{ceo}: kept {len(filtered_items)}/{len(original_items)} videos")